# CTAI 直肠癌 CT 肿瘤分割 — Kaggle GPU 训练

## 使用前准备
1. 将 `直肠癌数据/` 文件夹压缩为 zip 上传为 Kaggle Dataset（命名为 `ctai-rectal-cancer-ct`）
2. 将 `CTAI_model/` 代码文件夹压缩为 zip 上传为另一个 Kaggle Dataset（命名为 `ctai-model-code`）
3. 新建 Notebook → Settings → Accelerator → GPU T4 x2
4. Add Data → 添加上面两个 Dataset
5. 按顺序运行 Cell

In [ ]:
# Cell 1: 环境检查 + 安装依赖
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

!pip install -q albumentations SimpleITK tqdm pydicom scikit-image scipy

In [ ]:
# Cell 2: 复制代码到工作目录（Kaggle input 是只读的）
import os, shutil, glob

# 找到代码目录
code_candidates = [
    "/kaggle/input/ctai-model-code/CTAI_model",
    "/kaggle/input/ctai-model-code",
]
code_src = None
for c in code_candidates:
    if os.path.isfile(os.path.join(c, 'train.py')):
        code_src = c
        break

if code_src is None:
    # 自动搜索
    found = glob.glob('/kaggle/input/**/train.py', recursive=True)
    if found:
        code_src = os.path.dirname(found[0])
    else:
        raise FileNotFoundError('找不到代码目录，请检查 Dataset 名称')

work_dir = '/kaggle/working/CTAI_model'
if os.path.exists(work_dir):
    shutil.rmtree(work_dir)
shutil.copytree(code_src, work_dir)
os.chdir(work_dir)
print(f'代码目录: {work_dir}')
print(f'文件列表: {os.listdir(".")}')

# 检查数据目录
data_candidates = [
    "/kaggle/input/rectal-caner-data/rectal_data",
    "/kaggle/input/rectal-caner-data/直肠癌数据",
    "/kaggle/input/rectal-caner-data",
    "/kaggle/input/rectal-cancer-data/rectal_data",
]
data_dir = None
for d in data_candidates:
    if os.path.isdir(d):
        subs = os.listdir(d)
        # 检查是否有患者子目录（数字命名）
        if any(s.isdigit() for s in subs):
            data_dir = d
            break
if data_dir is None:
    # fallback: 搜索 dcm 文件
    dcm = glob.glob('/kaggle/input/**/*.dcm', recursive=True)
    if dcm:
        data_dir = os.path.dirname(os.path.dirname(os.path.dirname(dcm[0])))
    else:
        raise FileNotFoundError("找不到数据目录！")

patients = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
print(f'\n数据目录: {data_dir}')
print(f'患者数: {len(patients)}')

In [ ]:
# Cell 3: 检查数据目录
import glob

# 自动搜索数据
dcm_files = glob.glob('/kaggle/input/**/*.dcm', recursive=True)
print(f'找到 {len(dcm_files)} 个 DICOM 文件')
if dcm_files:
    print(f'示例路径: {dcm_files[0]}')
    # 推断数据根目录
    sample = dcm_files[0]
    # 路径格式: .../data_root/patient_id/arterial phase/xxx.dcm
    data_root = os.path.dirname(os.path.dirname(os.path.dirname(sample)))
    print(f'推断数据根目录: {data_root}')
    patients = [d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d))]
    print(f'患者数: {len(patients)}')

mask_files = glob.glob('/kaggle/input/**/*_mask.png', recursive=True)
print(f'找到 {len(mask_files)} 个 mask 文件')

In [ ]:
# Cell 4: 开始训练（全量数据，针对 Kaggle T4 优化参数）
# 
# 关键参数说明:
#   --batch_size 8      : T4 16GB 显存可以跑 8
#   --repeats 5         : 全量数据不需要大量重复
#   --epochs 100        : 全量数据 100 epoch 足够
#   --eval_interval 5   : 每 5 epoch 快速评估
#   --full_eval_interval 25 : 每 25 epoch 全量评估
#   --train_mode mixed  : 混合模式（肿瘤+部分背景切片）
#   --num_workers 2     : Kaggle 可用 2 个 worker
#   --val_max_slices 80 : 快速评估最多 80 张含肿瘤切片

!python train.py \
    --batch_size 8 \
    --repeats 5 \
    --epochs 100 \
    --eval_interval 5 \
    --full_eval_interval 25 \
    --train_mode mixed \
    --mixed_ratio 0.3 \
    --lr 3e-4 \
    --deep_supervision True \
    --use_ema True \
    --num_workers 2 \
    --val_max_slices 80 \
    --patience 10

In [ ]:
# Cell 5: 查看训练曲线
from IPython.display import Image, display
import os

curve_path = '/kaggle/working/logs/training_curves.png'
if os.path.exists(curve_path):
    display(Image(filename=curve_path))
else:
    print('训练曲线尚未生成')

In [ ]:
# Cell 6: 训练完成后 — 全量 TTA 评估
!python evaluate.py \
    --weights /kaggle/working/checkpoints/best_model.pth \
    --tta \
    --output_dir /kaggle/working/evaluation

In [ ]:
# Cell 7: 查看评估结果
import json
report_path = '/kaggle/working/evaluation/evaluation_report.json'
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)
    print('=== 评估摘要 ===')
    for k, v in report['summary'].items():
        print(f'  {k}: {v}')
else:
    print('评估报告不存在')

In [ ]:
# Cell 8: 打包模型下载
import shutil
shutil.make_archive('/kaggle/working/ctai_results', 'zip', '/kaggle/working/checkpoints')
print('模型已打包: /kaggle/working/ctai_results.zip')
print('可从 Output 标签页下载')